# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt
import rasterio
import numpy as np

import os

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

save_filtered_attributes = True

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
    print("Geometries cleaned")
    
    if save_filtered_attributes:
        if row['save_format'] == 'parquet':
            if not os.path.exists(f'{output_step2_path}/gpkg_attributs'):
                os.makedirs(f'{output_step2_path}/gpkg_attributs')
            gdf.to_parquet(f"{output_step2_path}/parquet_attributs/{attribute}.parquet")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        elif row['save_format'] == 'gpkg':
            if not os.path.exists(f'{output_step2_path}/parquet_attributs'):
                os.makedirs(f'{output_step2_path}/parquet_attributs')
            gdf.to_file(f"{output_step2_path}/gpkg_attributs/{attribute}.gpkg", driver="GPKG")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        else:
            if not os.path.exists(f'{output_step2_path}/csv_attributs'):
                os.makedirs(f'{output_step2_path}/csv_attributs')
            gdf.to_csv(f"{output_step2_path}/csv_attributs/{attribute}.csv", index=False)
            print(f"Warning: Unknown save format {row['save_format']} for attribute {attribute}. Data saved as csv.")
    else:
        print("Note : Save option is disabled.")

#### Import des attributs 

In [3]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

In [4]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,impact_attribut,proportional_weight_col_dtype,file_name,geometry_type,method,method_desc,how,value_column,buffer_size,filter_column,filter_values,crs,save_format
0,Agrément,vegetation,arbre_isole,True,arbre_isole,0.5,favorable,float,SIPV_ICA_ARBRE_ISOLE-SHP/SIPV_ICA_ARBRE_ISOLE.shp,point,A,Buffer du segment,sum,D_COURONNE,10,filtered,1,2056,parquet
1,Agrément,vegetation,espace_vert,True,domaine_routier,0.5,favorable,NaN,CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet
2,Sécurité,accident,accident,True,accident,0.3,defavorable,NaN,OTC_ACCIDENTS-SHP/OTC_ACCIDENTS.shp,point,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet
3,NaN,toilette,toilette,False,NaN,0.5,favorable,NaN,VDG_WC_PUBLIC-SHP/VDG_WC_PUBLIC.shp,point,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet
4,Sécurité,traffic,zone_apaisee,True,vitesse,0.5,favorable,NaN,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet
5,Sécurité,traffic,zone_pietonne,True,vitesse,1.0,favorable,NaN,OTC_ZONE_MODERATION_TRAFIC-SHP/OTC_ZONE_MODERA...,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet
6,Sécurité,traffic,vitesse,True,vitesse,0.6,defavorable,NaN,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet
7,Sécurité,largeur trottoir,ratio_trottoir,True,domaine_routier,0.5,favorable,float,CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet
8,Attractivité,eau,eau,True,eau,0.5,favorable,NaN,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet
9,Attractivité,proximite,rez_actif,True,rez_actif,0.5,favorable,NaN,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet


**Attribut Accidents**

In [5]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ANNEE > 2020, 'filtered'] = 1
gdf.loc[gdf.CONSEQ.isin(['Avec blessés graves', 'Avec tués']), 'filtered'] = 1
gdf.loc[gdf.VELOS == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_25 == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_45 == 1, 'filtered'] = 1
gdf.loc[gdf.PIETONS == 1, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: accident
Filters applied
Proportion of features accident kept after filtering:
filtered
0    0.582074
1    0.417926
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: accident in format: parquet


**Attribut Arbres isolés** (groupe Végétation)

In [6]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'arbre_isole'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: arbre_isole
Filters applied
Proportion of features arbre_isole kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: arbre_isole in format: parquet


**Attribut Espaces verts** (groupe Végétation)

In [7]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espace_vert'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.REVETEMENT.isin(['Arbustes', 'Terre', 'Gazon','Grille gazon']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espace_vert
Filters applied
Proportion of features espace_vert kept after filtering:
filtered
0    0.77455
1    0.22545
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: espace_vert in format: parquet


**Attribut Vitesse** (groupe Traffic)

In [8]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE == 0, 'filtered'] = 1

# Ne garder que les géométries du domaine routier qui intersectent les vitesses filtrées
domaine_routier = gpd.read_file(f'{input_file_path}/domaine_routier/CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp')
domaine_routier = domaine_routier.to_crs(gdf.crs)
chaussees = domaine_routier[domaine_routier['OBJET'] == 'Chaussée'].copy()


gdf.loc[gdf['TYPE_ZONE'] == 0, 'filtered'] = 1
gdf_filt = gdf[gdf['filtered'] == 1].copy()

# ⟶ garder uniquement les features de gdf qui intersectent une chaussée
gdf = (
    gpd.sjoin(chaussees, gdf, how="inner", predicate="intersects")
      .drop(columns="index_right")
      .copy()
)

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: vitesse
Filters applied
Proportion of features vitesse kept after filtering:
filtered
1    0.65339
0    0.34661
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: vitesse in format: parquet


**Attribut Zone pietonne** (groupe Traffic)

In [9]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_pietonne'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE.isin(['Zone piétonne', 'Zone de rencontre (20)', ]), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Filters applied
Proportion of features zone_pietonne kept after filtering:
filtered
0    0.536313
1    0.463687
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: zone_pietonne in format: parquet


/Users/Helo/miniconda3/envs/actionsituee/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(


**Attribut Zone apaisée** (groupe Traffic)

In [10]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_LIMIT.isin(['Zone 30 km/h', 'Prescription 30 km/h', 'Prescription 20 km/h']), 'filtered'] = 1 

# Ne garder que les géométries du domaine routier qui intersectent les vitesses filtrées
domaine_routier = gpd.read_file(f'{input_file_path}/domaine_routier/CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp')
domaine_routier = domaine_routier.to_crs(gdf.crs)
chaussees = domaine_routier[domaine_routier['OBJET'] == 'Chaussée'].copy()


gdf.loc[gdf['TYPE_ZONE'] == 0, 'filtered'] = 1
gdf_filt = gdf[gdf['filtered'] == 1].copy()

# ⟶ garder uniquement les features de gdf qui intersectent une chaussée
gdf = (
    gpd.sjoin(chaussees, gdf, how="inner", predicate="intersects")
      .drop(columns="index_right")
      .copy()
)
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_apaisee
Filters applied
Proportion of features zone_apaisee kept after filtering:
filtered
1    0.935563
0    0.064437
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: zone_apaisee in format: parquet


**Attribut Largeur trottoirs**

In [11]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'ratio_trottoir'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0


# Filter criterion
###----change here----------------------
gdf.loc[gdf.OBJET.isin(['Trottoir']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: ratio_trottoir
Filters applied
Proportion of features ratio_trottoir kept after filtering:
filtered
0    0.831949
1    0.168051
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: ratio_trottoir in format: parquet


**Eau**

In [12]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eau'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ETAT.isin(['A ciel ouvert']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Filters applied
Proportion of features eau kept after filtering:
filtered
1    0.615236
0    0.384764
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: eau in format: parquet


**Rez Actifs**

In [13]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#gdf.loc[gdf['BRANCHE'].str.contains('commerce de détail|détail|écoles|commerces|supermarchés|restaurants|banques|enseignement', case=False, na=False), 'filtered'] = 1
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: rez_actif
Filters applied
Proportion of features rez_actif kept after filtering:
filtered
0    0.86085
1    0.13915
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: rez_actif in format: parquet


**Attribut Bruit**

In [14]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bruit'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.CAT_J > 2, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: bruit
Filters applied
Proportion of features bruit kept after filtering:
filtered
0    0.887676
1    0.112324
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: bruit in format: parquet


**Attribut Proximité TP**

In [15]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tp'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: tp
Filters applied
Proportion of features tp kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: tp in format: parquet


**Attribut Stationnement Genants**

In [16]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_genant'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: stationnement_genant
Filters applied
Proportion of features stationnement_genant kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: stationnement_genant in format: parquet


**Attribut Proximité Aménités**

In [17]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: amenite
Filters applied
Proportion of features amenite kept after filtering:
filtered
0    0.86085
1    0.13915
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: amenite in format: parquet


**Attribut espaces ouverts**

In [18]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espaces_ouverts'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE == 'ESPACE PUBLIC', 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espaces_ouverts
Filters applied
Proportion of features espaces_ouverts kept after filtering:
filtered
0    0.589171
1    0.410829
Name: proportion, dtype: float64
Geometries cleaned
Filtered data saved for attribute: espaces_ouverts in format: parquet


**Attribut Confort thermique**

In [ ]:
# Initialize

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Sample every 10th point
    #rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nValue counts (first 10 most common values):")
print(gdf['temperature'].value_counts().head(10))
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
print(f"Saving {attribute}: ca take up to 3mn")
save(save_filtered_attributes, row, gdf, attribute)

Processing attribute: temperature

Value counts (first 10 most common values):
temperature
23.399401    319601
23.399500     75534
23.403999      4804
23.403700      4749
23.404100      4612
23.403900      4472
23.403799      4329
23.403601      4053
23.401600      3998
23.402599      3947
Name: count, dtype: int64

Descriptive statistics:
count    8.475526e+06
mean     2.796997e+01
std      3.924198e+00
min      1.726680e+01
25%      2.460520e+01
50%      2.902290e+01
75%      3.163050e+01
max      3.459120e+01
Name: temperature, dtype: float64
Geometries cleaned
Filtered data saved for attribute: temperature in format: parquet


In [20]:
#temp_parquet = gpd.read_parquet('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/parquet_attributs/temperature.parquet')
#temp_parquet.to_file('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs/temperature.gpkg', driver="GPKG")
